# Simple: сравнение DINOv2 и старой модели

Сравниваем модели на той же validation, затем один раз проверяем выбранный вариант на отложенной части train.csv. Для запуска нужен GPU в Colab и файлы, сохранённые после `simple_dinov2.ipynb`.

## 1. Подключаем данные и восстанавливаем разбиение

In [1]:
%pip install -q catboost iterative-stratification tqdm

from pathlib import Path
import hashlib
import json
import shutil
import zipfile

import numpy as np
import pandas as pd
import torch
from PIL import Image
from catboost import CatBoostClassifier
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import f1_score, precision_score, recall_score
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main')
if not (ROOT / 'data/train.csv').is_file():
    ROOT = Path(input('Путь к папке проекта с data/train.csv: ').strip())
assert (ROOT / 'data/train.csv').is_file(), ROOT
assets_zip = ROOT / 'simple_dinov2_comparison_assets.zip'
if assets_zip.is_file():
    with zipfile.ZipFile(assets_zip) as package:
        package.extractall(ROOT)
OUT = ROOT / 'labels/simple/models/dinov2'
ARCHIVE = ROOT / 'archive'
required = [
    OUT / 'validation_probabilities.csv',
    OUT / 'experiments.json',
    OUT / 'E2_cb_concat.cbm',
    OUT / 'E3_cb_mean_max_mesh.cbm',
    ROOT / 'labels/simple/models/simple_catboost_49.cbm',
    ARCHIVE / 'artifacts/internal_test_geometry5.csv',
    ARCHIVE / 'models/resnet/best_resnet18_improved.pth',
    ARCHIVE / 'models/catboost_features/simple.cbm',
    ARCHIVE / 'artifacts/resnet_catboost_features_config.json',
    ARCHIVE / 'artifacts/catboost_features_config.json',
    ARCHIVE / 'artifacts/train_mesh_features_relative.csv',
]
for path in required:
    assert path.is_file(), f'Не найден файл: {path}'

labels = pd.read_csv(ROOT / 'data/train.csv')
targets = labels.columns.drop('item_id').tolist()
assert labels.item_id.is_unique
split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_index, test_index = next(split.split(labels, labels[targets]))
train_val = labels.iloc[train_val_index].reset_index(drop=True)
test_df = labels.iloc[test_index].reset_index(drop=True)
split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1765, random_state=43)
train_index, val_index = next(split.split(train_val, train_val[targets]))
train_df = train_val.iloc[train_index].reset_index(drop=True)
val_df = train_val.iloc[val_index].reset_index(drop=True)
assert (len(train_df), len(val_df), len(test_df)) == (6259, 1361, 1344)
assert not (set(train_df.item_id) & set(val_df.item_id) |
            set(train_df.item_id) & set(test_df.item_id) |
            set(val_df.item_id) & set(test_df.item_id))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'Включите GPU в Colab'
torch.manual_seed(42)
np.random.seed(42)
print('Train:', len(train_df), 'Validation:', len(val_df), 'Internal test:', len(test_df))

Mounted at /content/drive
Train: 6259 Validation: 1361 Internal test: 1344


## 2. Загружаем PNG и mesh-признаки

In [2]:
TRAIN_DIR = ROOT / 'data/train'
png_zip = ROOT / 'data/train_png.zip'
if png_zip.is_file():
    local_zip = Path('/content/simple_dinov2_train_png.zip')
    if not local_zip.is_file() or local_zip.stat().st_size != png_zip.stat().st_size:
        shutil.copy2(png_zip, local_zip)
    local_images = Path('/content/simple_dinov2_png')
    marker = local_images / '.ready'
    if not marker.is_file():
        local_images.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(local_zip) as archive_file:
            for name in tqdm(archive_file.namelist(), desc='Распаковка PNG'):
                if name.lower().endswith('.png'):
                    archive_file.extract(name, local_images)
        marker.touch()
    first_png = next(local_images.rglob(f'{val_df.item_id.iloc[0]}.png'), None)
    assert first_png is not None, 'В архиве не найден PNG'
    TRAIN_DIR = first_png.parent
assert (TRAIN_DIR / f'{test_df.item_id.iloc[-1]}.png').is_file()

mesh_config = json.loads((ARCHIVE / 'artifacts/catboost_features_config.json').read_text(encoding='utf-8'))
mesh_columns = mesh_config['feature_columns']
assert mesh_config['target_columns'] == targets and len(mesh_columns) == 44
mesh_table = pd.read_csv(ARCHIVE / 'artifacts/train_mesh_features_relative.csv').set_index('item_id')
assert mesh_table.index.is_unique
geometry_columns = ['pca_thickness', 'pca_line_ratio', 'thickness_ratio',
                    'normal_share', 'surface_radius_cv']
extra_table = pd.read_csv(ARCHIVE / 'artifacts/internal_test_geometry5.csv').set_index('item_id')
assert extra_table.index.is_unique and set(extra_table.index) == set(test_df.item_id)
for frame in (val_df, test_df):
    assert np.isfinite(mesh_table.loc[frame.item_id, mesh_columns].to_numpy()).all()
print('PNG:', TRAIN_DIR, '| mesh-признаков:', len(mesh_columns))

Распаковка PNG:   0%|          | 0/8964 [00:00<?, ?it/s]

PNG: /content/simple_dinov2_png | mesh-признаков: 44


## 3. Получаем предсказания старой ResNet18 + CatBoost

In [3]:
image_transform = transforms.Compose([
    transforms.Resize((224, 336)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class RenderDataset(Dataset):
    def __init__(self, item_ids):
        self.item_ids = item_ids.tolist()

    def __len__(self):
        return len(self.item_ids)

    def __getitem__(self, index):
        with Image.open(TRAIN_DIR / f'{self.item_ids[index]}.png') as image:
            return image_transform(image.convert('RGB'))

checkpoint = torch.load(
    ARCHIVE / 'models/resnet/best_resnet18_improved.pth',
    map_location=device, weights_only=True
)
assert checkpoint['target_columns'] == targets
resnet = resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, len(targets))
resnet.load_state_dict(checkpoint['model_state_dict'])
resnet = resnet.to(device).eval()
simple_index = targets.index('simple')

def resnet_probabilities(frame):
    loader = DataLoader(RenderDataset(frame.item_id), batch_size=32, shuffle=False,
                        num_workers=0, pin_memory=True)
    blocks = []
    with torch.inference_mode():
        for images in tqdm(loader, desc='ResNet18'):
            blocks.append(torch.sigmoid(resnet(images.to(device)))[:, simple_index].cpu().numpy())
    return np.concatenate(blocks)

catboost = CatBoostClassifier()
catboost.load_model(str(ARCHIVE / 'models/catboost_features/simple.cbm'))
assert catboost.feature_names_ == mesh_columns
fusion_config = json.loads((ARCHIVE / 'artifacts/resnet_catboost_features_config.json').read_text(encoding='utf-8'))
assert fusion_config['target_columns'] == targets

def old_probabilities(frame):
    image_prob = resnet_probabilities(frame)
    mesh_prob = catboost.predict_proba(mesh_table.loc[frame.item_id, mesh_columns])[:, 1]
    fusion_prob = (fusion_config['resnet_weight'] * image_prob +
                   fusion_config['catboost_weight'] * mesh_prob)
    return image_prob, mesh_prob, fusion_prob

val_resnet, val_catboost, val_old = old_probabilities(val_df)
old_threshold = fusion_config['thresholds']['simple']
old_f1 = f1_score(val_df.simple, val_old >= old_threshold)
print('Старая связка F1:', round(old_f1, 4))
assert abs(old_f1 - 0.7090) < 0.0001, 'Старая модель не совпала с предыдущим результатом'
pd.DataFrame({'item_id': val_df.item_id, 'probability': val_resnet}).to_csv(
    OUT / 'resnet18_validation.csv', index=False)
pd.DataFrame({'item_id': val_df.item_id, 'probability': val_old}).to_csv(
    OUT / 'resnet_catboost_validation.csv', index=False)

ResNet18:   0%|          | 0/43 [00:00<?, ?it/s]

Старая связка F1: 0.709


## 4. Сравниваем ошибки и выбираем модель на validation

In [4]:
dino_table = pd.read_csv(OUT / 'validation_probabilities.csv').set_index('item_id')
assert dino_table.index.is_unique and set(dino_table.index) == set(val_df.item_id)
dino_table = dino_table.loc[val_df.item_id]
assert np.array_equal(dino_table.simple.to_numpy(), val_df.simple.to_numpy())
dino_config = json.loads((OUT / 'experiments.json').read_text(encoding='utf-8'))
assert dino_config['labels_sha256'] == hashlib.sha256((ROOT / 'data/train.csv').read_bytes()).hexdigest()
assert dino_config['train_size'] == len(train_df) and dino_config['validation_size'] == len(val_df)

dino_names = ['E2_cb_concat', 'E3_cb_mean_max_mesh', 'E4_geometry49']
val_probs = {'Old ResNet + CatBoost': val_old}
thresholds = {'Old ResNet + CatBoost': old_threshold}
for name in dino_names:
    val_probs[name] = dino_table[name].to_numpy()
    thresholds[name] = dino_config['experiments'][name]['Threshold']

y_val = val_df.simple.to_numpy()
def scores(y, p, threshold):
    prediction = p >= threshold
    return {'F1': f1_score(y, prediction),
            'Precision': precision_score(y, prediction, zero_division=0),
            'Recall': recall_score(y, prediction, zero_division=0),
            'FP': int(((y == 0) & prediction).sum()),
            'FN': int(((y == 1) & ~prediction).sum())}

rows = []
for name, p in val_probs.items():
    rows.append({'Модель': name, 'Порог': thresholds[name],
                 **scores(y_val, p, thresholds[name])})
display(pd.DataFrame(rows).round(4))

old_prediction = val_old >= old_threshold
dino_prediction = val_probs['E3_cb_mean_max_mesh'] >= thresholds['E3_cb_mean_max_mesh']
print('DINO исправила FN старой связки:', int(((y_val == 1) & ~old_prediction & dino_prediction).sum()))
print('DINO добавила FN:', int(((y_val == 1) & old_prediction & ~dino_prediction).sum()))
print('DINO исправила FP:', int(((y_val == 0) & old_prediction & ~dino_prediction).sum()))
print('DINO добавила FP:', int(((y_val == 0) & ~old_prediction & dino_prediction).sum()))

,Модель,Порог,F1,Precision,Recall,FP,FN
0,Old ResNet + CatBoost,0.43,0.7090,0.6415,0.7924,128,60
1,E2_cb_concat,0.31,0.7181,0.6590,0.7889,118,61
2,E3_cb_mean_max_mesh,0.28,0.7220,0.6492,0.8131,127,54
3,E4_geometry49,0.34,0.7229,0.6799,0.7716,105,66


DINO исправила FN старой связки: 15
DINO добавила FN: 9
DINO исправила FP: 27
DINO добавила FP: 26


In [5]:
# Подбираем только на validation; internal test в выборе не участвует.
grid = np.arange(0.05, 0.96, 0.01)
candidates = [{'name': name, 'dino': name if name in dino_names else None,
               'dino_weight': 1.0 if name in dino_names else 0.0,
               'threshold': float(thresholds[name]),
               'val_f1': scores(y_val, p, thresholds[name])['F1']}
              for name, p in val_probs.items()]
for name in dino_names[:2]:
    for weight in (0.25, 0.5, 0.75):
        p = weight * val_probs[name] + (1 - weight) * val_old
        threshold_scores = [f1_score(y_val, p >= t) for t in grid]
        best_index = int(np.argmax(threshold_scores))
        candidates.append({'name': f'{name} + old ({weight:.2f})',
                           'dino': name, 'dino_weight': weight,
                           'threshold': float(grid[best_index]),
                           'val_f1': float(threshold_scores[best_index])})
comparison = pd.DataFrame(candidates).sort_values('val_f1', ascending=False)
display(comparison.round(4))
selected = comparison.iloc[0].to_dict()
(OUT / 'selected_for_internal_test.json').write_text(
    json.dumps(selected, ensure_ascii=False, indent=2), encoding='utf-8')
print('Выбрано до проверки internal test:', selected)

,name,dino,dino_weight,threshold,val_f1
6,E2_cb_concat + old (0.75),E2_cb_concat,0.75,0.35,0.7261
7,E3_cb_mean_max_mesh + old (0.25),E3_cb_mean_max_mesh,0.25,0.39,0.7245
8,E3_cb_mean_max_mesh + old (0.50),E3_cb_mean_max_mesh,0.50,0.40,0.7240
5,E2_cb_concat + old (0.50),E2_cb_concat,0.50,0.33,0.7240
3,E4_geometry49,E4_geometry49,1.00,0.34,0.7229
9,E3_cb_mean_max_mesh + old (0.75),E3_cb_mean_max_mesh,0.75,0.27,0.7227
2,E3_cb_mean_max_mesh,E3_cb_mean_max_mesh,1.00,0.28,0.7220
4,E2_cb_concat + old (0.25),E2_cb_concat,0.25,0.40,0.7219
1,E2_cb_concat,E2_cb_concat,1.00,0.31,0.7181
0,Old ResNet + CatBoost,None,0.00,0.43,0.7090


Выбрано до проверки internal test: {'name': 'E2_cb_concat + old (0.75)', 'dino': 'E2_cb_concat', 'dino_weight': 0.75, 'threshold': 0.35000000000000003, 'val_f1': 0.7261146496815286}


## 5. Считаем предсказания на отложенных объектах

In [6]:
test_resnet, test_catboost, test_old = old_probabilities(test_df)
del resnet, catboost
torch.cuda.empty_cache()

def split_views(image):
    image = image.convert('RGB')
    width, height = image.size
    assert width % 3 == 0 and height % 2 == 0 and width // 3 == height // 2
    side = width // 3
    return [image.crop((col * side, row * side, (col + 1) * side, (row + 1) * side))
            for row in range(2) for col in range(3)]

dino_transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC,
                      antialias=True),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class SixViewsDataset(Dataset):
    def __len__(self):
        return len(test_df)

    def __getitem__(self, index):
        item_id = test_df.item_id.iloc[index]
        with Image.open(TRAIN_DIR / f'{item_id}.png') as image:
            return torch.stack([dino_transform(view) for view in split_views(image)])

embed_path = OUT / 'internal_test_embeddings.npy'
manifest_path = OUT / 'internal_test_embeddings.json'
manifest = {'model': 'dinov2_vits14', 'revision': '7764ea0f912e53c92e82eb78a2a1631e92725fc8',
            'item_ids': test_df.item_id.tolist(),
            'labels_sha256': hashlib.sha256((ROOT / 'data/train.csv').read_bytes()).hexdigest()}
if embed_path.is_file() and manifest_path.is_file():
    assert json.loads(manifest_path.read_text(encoding='utf-8')) == manifest
    test_embeddings = np.load(embed_path)
    print('Загружен кеш DINOv2')
else:
    import os
    os.environ['XFORMERS_DISABLED'] = '1'
    torch.hub.set_dir(str(OUT / 'hub'))
    dino = torch.hub.load('facebookresearch/dinov2:7764ea0f912e53c92e82eb78a2a1631e92725fc8',
                          'dinov2_vits14', pretrained=True, trust_repo=True, skip_validation=True)
    dino = dino.to(device).eval()
    dino.requires_grad_(False)
    test_embeddings = np.empty((len(test_df), 6, 384), dtype=np.float32)
    loader = DataLoader(SixViewsDataset(), batch_size=2, shuffle=False,
                        num_workers=0, pin_memory=True)
    position = 0
    with torch.inference_mode():
        for views in tqdm(loader, desc='DINOv2 internal test'):
            flat = views.reshape(-1, 3, 224, 224)
            parts = []
            for start in range(0, len(flat), 12):
                parts.append(dino(flat[start:start + 12].to(device)).float().cpu().numpy())
            batch = np.concatenate(parts).reshape(len(views), 6, 384)
            test_embeddings[position:position + len(views)] = batch
            position += len(views)
    assert position == len(test_df) and np.isfinite(test_embeddings).all()
    np.save(embed_path, test_embeddings)
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False), encoding='utf-8')
    del dino
    torch.cuda.empty_cache()
assert test_embeddings.shape == (len(test_df), 6, 384)

ResNet18:   0%|          | 0/42 [00:00<?, ?it/s]

Using cache found in /content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main/labels/simple/models/dinov2/hub/facebookresearch_dinov2_7764ea0f912e53c92e82eb78a2a1631e92725fc8
/content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main/labels/simple/models/dinov2/hub/facebookresearch_dinov2_7764ea0f912e53c92e82eb78a2a1631e92725fc8/dinov2/layers/swiglu_ffn.py:45: UserWarning: xFormers is disabled (SwiGLU)
  warnings.warn("xFormers is disabled (SwiGLU)")
/content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main/labels/simple/models/dinov2/hub/facebookresearch_dinov2_7764ea0f912e53c92e82eb78a2a1631e92725fc8/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main/labels/simple/models/dinov2/hub/facebookresearch_dinov2_7764ea0f912e53c92e82eb78a2a1631e92725fc8/dinov2/layers/attention.py:29: UserWarning: xFormers 

DINOv2 internal test:   0%|          | 0/672 [00:00<?, ?it/s]

In [7]:
test_probs = {'Old ResNet + CatBoost': test_old}
for name in dino_names[:2]:
    model = CatBoostClassifier()
    model.load_model(str(OUT / f'{name}.cbm'))
    if name == 'E2_cb_concat':
        features = test_embeddings.reshape(len(test_df), 6 * 384)
    else:
        views_mean = test_embeddings.mean(axis=1)
        views_max = test_embeddings.max(axis=1)
        mesh = mesh_table.loc[test_df.item_id, mesh_columns].to_numpy(dtype=np.float32)
        features = np.concatenate([views_mean, views_max, mesh], axis=1)
    test_probs[name] = model.predict_proba(features)[:, 1]
    del model, features

geometry_model = CatBoostClassifier()
geometry_model.load_model(str(ROOT / 'labels/simple/models/simple_catboost_49.cbm'))
geometry_features = mesh_table.loc[test_df.item_id, mesh_columns].join(
    extra_table[geometry_columns], validate='one_to_one')
assert geometry_model.feature_names_ == mesh_columns + geometry_columns
geometry_prob = geometry_model.predict_proba(geometry_features)[:, 1]
e4_config = dino_config['experiments']['E4_geometry49']
e4_weight = e4_config['dino_weight']
assert e4_config['dino_experiment'] == 'E2_cb_concat'
test_probs['E4_geometry49'] = (e4_weight * test_probs['E2_cb_concat'] +
                                (1 - e4_weight) * geometry_prob)

if selected['dino'] is None:
    selected_prob = test_old
else:
    w = selected['dino_weight']
    selected_prob = w * test_probs[selected['dino']] + (1 - w) * test_old

# Только здесь впервые смотрим метки internal test.
y_test = test_df.simple.to_numpy()
test_rows = []
for name in test_probs:
    test_rows.append({'Модель': name, 'Порог': thresholds[name],
                      **scores(y_test, test_probs[name], thresholds[name])})
if selected['name'] not in test_probs:
    test_rows.append({'Модель': selected['name'], 'Порог': selected['threshold'],
                      **scores(y_test, selected_prob, selected['threshold'])})
test_result = pd.DataFrame(test_rows)
display(test_result.round(4))
test_result.to_csv(OUT / 'internal_test_comparison.csv', index=False)
pd.DataFrame({'item_id': test_df.item_id,
              'old_probability': test_old,
              'dino_visual_probability': test_probs['E2_cb_concat'],
              'dino_mesh_probability': test_probs['E3_cb_mean_max_mesh'],
              'selected_probability': selected_prob}).to_csv(
    OUT / 'internal_test_probabilities.csv', index=False)
print('Результаты сохранены в', OUT)

,Модель,Порог,F1,Precision,Recall,FP,FN
0,Old ResNet + CatBoost,0.43,0.7017,0.6341,0.7855,131,62
1,E2_cb_concat,0.31,0.7079,0.6540,0.7716,118,66
2,E3_cb_mean_max_mesh,0.28,0.6991,0.6233,0.7958,139,59
3,E4_geometry49,0.34,0.7282,0.6950,0.7647,97,68
4,E2_cb_concat + old (0.75),0.35,0.7223,0.6737,0.7785,109,64


Результаты сохранены в /content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main/labels/simple/models/dinov2
